# Seed search — AC(128) reaching 2000 hits + MP4

This notebook:
1. Searches for a random seed where the **Actor-Critic (hidden_dim=128)** can reach **TARGET_HITS=2000** in a single deterministic rollout.
2. Records that rollout into an **.mp4** using `ffmpeg`.

Notes:
- Seed search is run **without rendering** for speed.
- Recording captures frames every `FRAME_SKIP` steps to keep the MP4 size manageable.
- Make sure the checkpoint exists at `CKPT_PATH`.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import subprocess
import sys

import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'run').exists() and (PROJECT_ROOT.parent / 'run').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.environment.pong_env import PongEnv
from src.environment.renderer import PongRenderer
from run.eval import load_agent, get_deterministic_action

# ------------------- user params -------------------
HIDDEN_DIM = 256
AGENT = 'actor_critic'
CKPT_PATH = PROJECT_ROOT / 'artifacts' / 'actor_critic_model.pt'

TARGET_HITS = 1500
MAX_STEPS = 2_000_000  # safety cap

SEED_START = 0
SEED_END = 5000  # increase if needed

FRAME_SKIP = 4  # capture every N env steps
FPS = 30

# MP4 encoding settings
# - CRF: lower => higher quality (18 is visually lossless for most content)
# - preset: slower => better compression at same quality
CRF = 18
PRESET = 'slow'

# Upscale before encoding to avoid blurry playback (nearest-neighbor keeps pixel-art crisp)
UPSCALE = 8  # output resolution = (W*UPSCALE) x (H*UPSCALE)

# When we stop (reach TARGET_HITS), keep the final frame with an overlay for a bit
WIN_TEXT = 'YOU WON'
WIN_HOLD_SECONDS = 2

OUTDIR = PROJECT_ROOT / 'artifacts' / 'seed_search'
OUTDIR.mkdir(parents=True, exist_ok=True)
MP4_PATH = OUTDIR / f'{AGENT}_hd{HIDDEN_DIM}_hits{TARGET_HITS}.mp4'
# ---------------------------------------------------

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
print('Checkpoint:', CKPT_PATH)
print('MP4 output :', MP4_PATH)


pygame-ce 2.5.7 (SDL 2.32.10, Python 3.14.0)
Checkpoint: /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/actor_critic_model.pt
MP4 output : /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/seed_search/actor_critic_hd256_hits1500.mp4


In [2]:
def rollout_hits(seed: int) -> tuple[int, int]:
    """Return (hits, steps) for one deterministic rollout."""
    env = PongEnv()
    env.seed(seed)
    env.set_random_bounce(True)
    env.set_global_step(200_000)  # hardest opponent regime used in eval

    agent = load_agent(AGENT, str(CKPT_PATH), device='cpu', hidden_dim_override=HIDDEN_DIM)

    state = env.reset()
    steps = 0
    hits = 0
    done = False

    while not done and steps < MAX_STEPS:
        action = get_deterministic_action(agent, state)
        state, reward, terminated, truncated, info = env.step(action)
        steps += 1
        hits = int(info.get('hits', hits))
        done = bool(terminated) or hits >= TARGET_HITS

    return hits, steps


best = {'seed': None, 'hits': -1, 'steps': None}
found_seed = None

for seed in range(SEED_START, SEED_END):
    hits, steps = rollout_hits(seed)
    if hits > best['hits']:
        best = {'seed': seed, 'hits': hits, 'steps': steps}
        print(f"[best] seed={seed} hits={hits} steps={steps}")

    if hits >= TARGET_HITS:
        found_seed = seed
        print(f"[FOUND] seed={seed} hits={hits} steps={steps}")
        break

print('Best so far:', best)
assert found_seed is not None, 'No seed found in the given range. Increase SEED_END.'


Loading checkpoint from: /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/actor_critic_model.pt
[best] seed=0 hits=1500 steps=71964
[FOUND] seed=0 hits=1500 steps=71964
Best so far: {'seed': 0, 'hits': 1500, 'steps': 71964}


In [3]:
# Record MP4 for the found seed
frames_dir = OUTDIR / 'frames_tmp'
if frames_dir.exists():
    shutil.rmtree(frames_dir)
frames_dir.mkdir(parents=True, exist_ok=True)

env = PongEnv()
env.seed(found_seed)
env.set_random_bounce(True)
env.set_global_step(200_000)
agent = load_agent(AGENT, str(CKPT_PATH), device='cpu', hidden_dim_override=HIDDEN_DIM)

renderer = PongRenderer()
state = env.reset()
steps = 0
hits = 0
frame_idx = 0
done = False

while not done and steps < MAX_STEPS:
    action = get_deterministic_action(agent, state)
    state, reward, terminated, truncated, info = env.step(action)
    steps += 1
    hits = int(info.get('hits', hits))

    if steps % FRAME_SKIP == 0:
        frame = renderer.capture_frame(
            bx=env.bx,
            by=env.by,
            py_agent=env.py,
            py_opponent=env.ly,
            score_agent=hits,
            score_opponent=0,
        )
        Image.fromarray(frame).save(frames_dir / f'frame_{frame_idx:06d}.png')
        frame_idx += 1

    done = bool(terminated) or hits >= TARGET_HITS

# Hold the final state and overlay a win message
hold_frames = int(FPS * WIN_HOLD_SECONDS)
for _ in range(hold_frames):
    frame = renderer.capture_frame(
        bx=env.bx,
        by=env.by,
        py_agent=env.py,
        py_opponent=env.ly,
        score_agent=hits,
        score_opponent=0,
        overlay_text=WIN_TEXT,
    )
    Image.fromarray(frame).save(frames_dir / f'frame_{frame_idx:06d}.png')
    frame_idx += 1

renderer.close()

print(f"Recorded frames: {frame_idx} (steps={steps}, hits={hits}, seed={found_seed})")
assert hits >= TARGET_HITS, f"Recording ended before reaching target hits: {hits} < {TARGET_HITS}"

# Encode MP4 via ffmpeg
ffmpeg = shutil.which('ffmpeg')
assert ffmpeg, 'ffmpeg not found in PATH'

scale_filter = f"scale=iw*{UPSCALE}:ih*{UPSCALE}:flags=neighbor"

cmd = [
    ffmpeg,
    '-y',
    '-framerate', str(FPS),
    '-i', str(frames_dir / 'frame_%06d.png'),
    '-vf', scale_filter,
    '-c:v', 'libx264',
    '-preset', str(PRESET),
    '-crf', str(CRF),
    '-pix_fmt', 'yuv420p',
    '-movflags', '+faststart',
    str(MP4_PATH),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

print('Saved MP4:', MP4_PATH)


Loading checkpoint from: /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/actor_critic_model.pt
Recorded frames: 18051 (steps=71964, hits=1500, seed=0)
Running: /opt/homebrew/bin/ffmpeg -y -framerate 30 -i /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/seed_search/frames_tmp/frame_%06d.png -vf scale=iw*8:ih*8:flags=neighbor -c:v libx264 -preset slow -crf 18 -pix_fmt yuv420p -movflags +faststart /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/seed_search/actor_critic_hd256_hits1500.mp4


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, image2, from '/Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/seed_search/frames_tmp/frame_%06d.png':
  Duration: 00:10:01.70, start: 0.000000, bitrate: N/A
  Stream #0:0: Video: png

Saved MP4: /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/seed_search/actor_critic_hd256_hits1500.mp4


[mp4 @ 0x942c14280] Starting second pass: moving the moov atom to the beginning of the file5.1x elapsed=0:00:13.12    
[out#0/mp4 @ 0x943400780] video:1554KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 11.854393%
frame=18051 fps=1357 q=-1.0 Lsize=    1738KiB time=00:10:01.63 bitrate=  23.7kbits/s speed=45.2x elapsed=0:00:13.30    
[libx264 @ 0x943018a80] frame I:73    Avg QP: 3.92  size:   853
[libx264 @ 0x943018a80] frame P:9601  Avg QP:11.69  size:   107
[libx264 @ 0x943018a80] frame B:8377  Avg QP:13.93  size:    60
[libx264 @ 0x943018a80] consecutive B-frames: 28.6% 23.7% 15.1% 32.7%
[libx264 @ 0x943018a80] mb I  I16..4: 96.3%  0.5%  3.2%
[libx264 @ 0x943018a80] mb P  I16..4:  1.2%  0.3%  0.1%  P16..4:  0.3%  0.1%  0.1%  0.0%  0.0%    skip:97.8%
[libx264 @ 0x943018a80] mb B  I16..4:  0.1%  0.0%  0.0%  B16..8:  1.8%  0.1%  0.0%  direct: 0.0%  skip:97.9%  L0:48.8% L1:51.2% BI: 0.0%
[libx264 @ 0x943018a80] 8x8 transform intra:13.2% inter:99.2%
[li